# 시간 지연 탐구 (Colab)
속도 → 뮤온 → 중력/GPS → 슈뢰딩거 → 시간의 문제

> 그래프 글자는 코랩에 한글 폰트가 없어 영어로 표기했습니다.

In [ ]:
# 0. 저장소에서 physics.py 불러오기 (REPO_URL을 자신의 저장소로 바꾸세요)
REPO_URL = "https://github.com/<아이디>/time-dilation-explorer"
import os, sys
if not os.path.exists("time-dilation-explorer"):
    !git clone {REPO_URL}
sys.path.append("time-dilation-explorer")
import numpy as np, matplotlib.pyplot as plt
import physics as ph

## 1. 속도와 시간: γ 곡선과 일상 속도

In [ ]:
beta = np.linspace(0, 0.999, 500)
plt.plot(beta, ph.gamma(beta)); plt.xlabel("v / c"); plt.ylabel("gamma"); plt.title("Time dilation factor"); plt.show()

things = {"walking": 1.4, "jet": 250, "ISS": 7660, "muon 0.99c": 0.99*ph.C}
for k, v in things.items():
    print(f"{k:12s} gamma-1 = {float(ph.gamma_minus_one(v/ph.C)):.3e}   lag per 1 s = {float(ph.clock_lag(v)):.3e} s")

## 2. 뮤온 몬테카를로
거리·속도를 바꿔 보고, 원 논문(Frisch & Smith 1963)의 값과 비교해 보세요.

In [ ]:
d, b, n = 1907, 0.994, 20000
no = ph.simulate_muons(n, d, b, dilation=False)
yes = ph.simulate_muons(n, d, b, dilation=True)
print("without dilation:", no, " with dilation:", yes)
print("analytic fraction:", ph.muon_survival_fraction(d, b, False), ph.muon_survival_fraction(d, b, True))
plt.bar(["no dilation", "dilation"], [no, yes]); plt.ylabel("muons reaching ground"); plt.show()

## 3. 중력 시간 지연과 GPS

In [ ]:
alt = np.linspace(300, 40000, 300)
off = ph.gps_clock_offsets(alt*1e3 + ph.R_EARTH)
plt.plot(alt, off["sr_us"], label="velocity"); plt.plot(alt, off["gr_us"], label="gravity"); plt.plot(alt, off["net_us"], label="net")
plt.axvline(20200, ls="--", c="gray"); plt.xlabel("altitude (km)"); plt.ylabel("us / day"); plt.legend(); plt.show()
g = ph.gps_clock_offsets(); print({k: round(float(v), 2) for k, v in g.items()})
for name, (m, r) in ph.PRESETS.items():
    print(name, float(ph.static_clock_rate(m, r)))

## 4. 슈뢰딩거 방정식: 시간은 바깥에서 주어진다

In [ ]:
x = ph.make_grid()
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, kind in zip(axes, ["free", "harmonic", "barrier"]):
    frames = ph.evolve(ph.gaussian_packet(x), x, ph.potential(x, kind), steps=500)
    for i in [0, len(frames)//3, 2*len(frames)//3, len(frames)-1]:
        ax.plot(x, frames[i], label=f"frame {i}")
    ax.set_title(kind); ax.set_xlim(-45, 45)
axes[0].legend(); plt.show()

## 5. 시간의 문제: Page–Wootters 장난감 모형

In [ ]:
out = ph.page_wootters_toy(n_clock=16, omega=1.0)
plt.plot(out["times"], out["cond_z"], "o", label="conditioned on clock")
plt.plot(out["times"], out["expected_z"], "-", label="Schrodinger solution"); plt.legend(); plt.xlabel("clock reading"); plt.show()
print("<sz> without clock:", out["z_without_clock"], "  entanglement (bits):", out["entropy_bits"])